<a href="https://colab.research.google.com/github/spdb9876/loan-default-prediction/blob/main/01_data_wrangling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from google.colab import userdata

# Step 1: Install the Kaggle API client
!pip install kaggle --quiet

# Retrieve the Kaggle API key from Colab secrets
# retrive from enviornment variable
kaggle_json_content = userdata.get('KAGGLE_JSON')

# Create the ~/.kaggle directory if it doesn't exist
os.makedirs('/root/.kaggle', exist_ok=True)

# Write the content to kaggle.json
with open('/root/.kaggle/kaggle.json', 'w') as f:
    f.write(kaggle_json_content)

# Set correct permissions (important for security)
os.chmod('/root/.kaggle/kaggle.json', 600)

print("Kaggle API key setup complete using Colab Secrets.")

# Step 3: Download the specified file from the competition
competition_name = "home-credit-default-risk"
# Correcting the file name: often Kaggle uses underscores instead of periods.
file_name = "application_train.csv"
output_directory = "."

print(f"Attempting to download {file_name} from {competition_name}...")
!kaggle competitions download -c {competition_name} -f {file_name} -p {output_directory}


print(f"Download command executed. Check the directory '{output_directory}' for the file.")

Kaggle API key setup complete using Colab Secrets.
Attempting to download application_train.csv from home-credit-default-risk...
100% 36.1M/36.1M [00:00<00:00, 141MB/s] 

Download command executed. Check the directory '.' for the file.


In [ ]:
import zipfile
import pandas as pd

# Unzip the downloaded file
zip_file_name = f"./{file_name}.zip"
with zipfile.ZipFile(zip_file_name, 'r') as zip_ref:
    zip_ref.extractall(output_directory)
print(f"Extracted {file_name} from {zip_file_name} to {output_directory}/")

# List files in the directory to confirm extraction
!ls -lh {output_directory}

Extracted application_train.csv from ./application_train.csv.zip to ./
total 195M
-rw-r--r-- 1 root root 159M Mar 29 09:55 application_train.csv
-rw-r--r-- 1 root root  37M Dec 11  2019 application_train.csv.zip
drwxr-xr-x 1 root root 4.0K Mar 23 13:29 sample_data


In [ ]:
# Load the extracted CSV file into a pandas DataFrame
df = pd.read_csv(f"{output_directory}/{file_name}")

# Display the first 5 rows of the DataFrame
print(f"\nDisplaying the first 5 rows of {file_name}:")
display(df.head())


Displaying the first 5 rows of application_train.csv:


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0




```
# This is formatted as code
```



In [ ]:
missing_data = df.isnull().sum()
missing_percentage = (missing_data / len(df)) * 100

# Create a DataFrame for missing values
missing_info = pd.DataFrame({
    'Missing Count': missing_data,
    'Missing Percentage': missing_percentage
})

# Filter out columns with no missing values and sort by percentage
missing_info = missing_info[missing_info['Missing Count'] > 0].sort_values(by='Missing Percentage', ascending=False)

print("\nMissing Data Analysis for application_train.csv:")
display(missing_info)


Missing Data Analysis for application_train.csv:


,Missing Count,Missing Percentage
COMMONAREA_MEDI,214865,69.872297
COMMONAREA_MODE,214865,69.872297
COMMONAREA_AVG,214865,69.872297
NONLIVINGAPARTMENTS_MODE,213514,69.432963
NONLIVINGAPARTMENTS_MEDI,213514,69.432963
...,...,...
EXT_SOURCE_2,660,0.214626
AMT_GOODS_PRICE,278,0.090403
AMT_ANNUITY,12,0.003902
CNT_FAM_MEMBERS,2,0.000650


### Strategies for Handling Missing Values

When dealing with missing values, the strategy often depends on the percentage of missing data and the nature of the column (numerical or categorical).

1.  **Drop Columns with High Missing Percentage**: For columns with a very high percentage of missing values (e.g., over 60-70%), it's often best to drop them as imputing them might introduce too much noise or bias. These columns might not provide much predictive power.
    *   **Example from `missing_info`**: `COMMONAREA_MEDI`, `COMMONAREA_MODE`, `COMMONAREA_AVG`, `NONLIVINGAPARTMENTS_MODE`, etc., all have around 70% missing data.

2.  **Imputation for Numerical Columns (Moderate Missing Percentage)**:
    *   **Median/Mean Imputation**: For numerical features with a moderate amount of missing data, imputing with the median (less sensitive to outliers) or mean can be effective.
    *   **Example**: Columns like `AMT_REQ_CREDIT_BUREAU_DAY` (around 13.5% missing) or `EXT_SOURCE_1` (around 56% missing) could be candidates for this, depending on domain knowledge and further analysis.

3.  **Imputation for Categorical Columns (Moderate Missing Percentage)**:
    *   **Mode Imputation**: For categorical features, imputing with the mode (most frequent category) is a common practice.
    *   **Treat as a Separate Category**: Sometimes, the fact that a value is missing can itself be informative. In such cases, you can treat missing values as a new, distinct category.

4.  **Drop Rows (Low Missing Percentage)**: If a column has a very small percentage of missing values (e.g., less than 5%) and dropping these few rows won't significantly reduce your dataset size, it can be a simple and effective strategy, especially if imputation is complex or inappropriate for the feature.
    *   **Example**: `EXT_SOURCE_2` has only 0.21% missing data, making it a candidate for row dropping if `SK_ID_CURR` is the unique identifier for rows and imputation is not preferred.

---

### Applying Missing Value Strategies

Let's apply some of these strategies. We'll start by dropping columns with more than 60% missing values.

In [ ]:
# Get columns with more than 60% missing values
threshold = 60
columns_to_drop = missing_info[missing_info['Missing Percentage'] > threshold].index.tolist()

print(f"Dropping {len(columns_to_drop)} columns with more than {threshold}% missing values:")
print(columns_to_drop)

# Drop these columns from the DataFrame
df_cleaned = df.drop(columns=columns_to_drop)

print(f"DataFrame shape after dropping columns: {df_cleaned.shape}")


Dropping 17 columns with more than 60% missing values:
['COMMONAREA_MEDI', 'COMMONAREA_MODE', 'COMMONAREA_AVG', 'NONLIVINGAPARTMENTS_MODE', 'NONLIVINGAPARTMENTS_MEDI', 'NONLIVINGAPARTMENTS_AVG', 'FONDKAPREMONT_MODE', 'LIVINGAPARTMENTS_AVG', 'LIVINGAPARTMENTS_MEDI', 'LIVINGAPARTMENTS_MODE', 'FLOORSMIN_MEDI', 'FLOORSMIN_MODE', 'FLOORSMIN_AVG', 'YEARS_BUILD_MODE', 'YEARS_BUILD_MEDI', 'YEARS_BUILD_AVG', 'OWN_CAR_AGE']
DataFrame shape after dropping columns: (307511, 105)


Next, let's impute remaining numerical columns with their median and categorical columns with their mode. We'll recalculate missing values for the `df_cleaned` DataFrame first.

In [ ]:
# Recalculate missing info for the cleaned DataFrame
missing_data_cleaned = df_cleaned.isnull().sum()
missing_percentage_cleaned = (missing_data_cleaned / len(df_cleaned)) * 100

missing_info_cleaned = pd.DataFrame({
    'Missing Count': missing_data_cleaned,
    'Missing Percentage': missing_percentage_cleaned
})

missing_info_cleaned = missing_info_cleaned[missing_info_cleaned['Missing Count'] > 0].sort_values(by='Missing Percentage', ascending=False)

print("\nMissing Data Analysis for df_cleaned (after dropping high-missing columns):")
display(missing_info_cleaned)

# Identify numerical and categorical columns with missing values for imputation
numerical_cols_with_missing = df_cleaned[missing_info_cleaned.index].select_dtypes(include=['int64', 'float64']).columns
categorical_cols_with_missing = df_cleaned[missing_info_cleaned.index].select_dtypes(include=['object', 'category']).columns

# Impute numerical columns with median
for col in numerical_cols_with_missing:
    if df_cleaned[col].isnull().any():
        median_val = df_cleaned[col].median()
        df_cleaned[col].fillna(median_val, inplace=True)
        print(f"Imputed numerical column '{col}' with median: {median_val}")

# Impute categorical columns with mode
for col in categorical_cols_with_missing:
    if df_cleaned[col].isnull().any():
        mode_val = df_cleaned[col].mode()[0] # .mode() can return multiple values, take the first
        df_cleaned[col].fillna(mode_val, inplace=True)
        print(f"Imputed categorical column '{col}' with mode: {mode_val}")

print("\nMissing value imputation complete. Verifying no missing values remain:")
print(df_cleaned.isnull().sum().sum()) # Should be 0 if all missing values are handled



Missing Data Analysis for df_cleaned (after dropping high-missing columns):


,Missing Count,Missing Percentage
LANDAREA_AVG,182590,59.376738
LANDAREA_MODE,182590,59.376738
LANDAREA_MEDI,182590,59.376738
BASEMENTAREA_MEDI,179943,58.515956
BASEMENTAREA_MODE,179943,58.515956
BASEMENTAREA_AVG,179943,58.515956
EXT_SOURCE_1,173378,56.381073
NONLIVINGAREA_AVG,169682,55.179164
NONLIVINGAREA_MEDI,169682,55.179164
NONLIVINGAREA_MODE,169682,55.179164


/tmp/ipykernel_1827/1260736512.py:23: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_cleaned[col].fillna(median_val, inplace=True)


Imputed numerical column 'LANDAREA_AVG' with median: 0.0481
Imputed numerical column 'LANDAREA_MODE' with median: 0.0458
Imputed numerical column 'LANDAREA_MEDI' with median: 0.0487
Imputed numerical column 'BASEMENTAREA_MEDI' with median: 0.0758
Imputed numerical column 'BASEMENTAREA_MODE' with median: 0.0746
Imputed numerical column 'BASEMENTAREA_AVG' with median: 0.0763
Imputed numerical column 'EXT_SOURCE_1' with median: 0.5059979305057544
Imputed numerical column 'NONLIVINGAREA_AVG' with median: 0.0036
Imputed numerical column 'NONLIVINGAREA_MEDI' with median: 0.0031
Imputed numerical column 'NONLIVINGAREA_MODE' with median: 0.0011
Imputed numerical column 'ELEVATORS_MEDI' with median: 0.0
Imputed numerical column 'ELEVATORS_MODE' with median: 0.0
Imputed numerical column 'ELEVATORS_AVG' with median: 0.0
Imputed numerical column 'APARTMENTS_AVG' with median: 0.0876
Imputed numerical column 'APARTMENTS_MODE' with median: 0.084
Imputed numerical column 'APARTMENTS_MEDI' with median:

/tmp/ipykernel_1827/1260736512.py:30: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_cleaned[col].fillna(mode_val, inplace=True)


Imputed categorical column 'HOUSETYPE_MODE' with mode: block of flats
Imputed categorical column 'EMERGENCYSTATE_MODE' with mode: No
Imputed categorical column 'OCCUPATION_TYPE' with mode: Laborers
Imputed categorical column 'NAME_TYPE_SUITE' with mode: Unaccompanied

Missing value imputation complete. Verifying no missing values remain:
0


In [ ]:
print("Total missing values remaining in df_cleaned:", df_cleaned.isnull().sum().sum())

Total missing values remaining in df_cleaned: 0
